<a href="https://colab.research.google.com/github/sajidazahir/Devops-automation/blob/main/Sajida_AI_samples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
df = pd.read_csv("/content/sample_data/california_housing_train.csv")


In [ ]:
df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0
...,...,...,...,...,...,...,...,...,...
16995,-124.26,40.58,52.0,2217.0,394.0,907.0,369.0,2.3571,111400.0
16996,-124.27,40.69,36.0,2349.0,528.0,1194.0,465.0,2.5179,79000.0
16997,-124.30,41.84,17.0,2677.0,531.0,1244.0,456.0,3.0313,103600.0
16998,-124.30,41.80,19.0,2672.0,552.0,1298.0,478.0,1.9797,85800.0


In [1]:
print("Hello World..!")

Hello World..!


## 1. Data Preparation: Loading a Sample Text Dataset

To begin building our simple LLM, we need a corpus of text data. We'll use the `nltk` library to access a classic text corpus, specifically from the Gutenberg project, which contains many free e-books.

In [8]:
# Install NLTK if you haven't already
!pip install -qqq nltk
import nltk

# Download the 'gutenberg' corpus
nltk.download('gutenberg', quiet=True)

from nltk.corpus import gutenberg

# Let's pick a book from the Gutenberg corpus, for example, Shakespeare's 'hamlet'
hamlet_text = gutenberg.raw('shakespeare-hamlet.txt')

print(f"Loaded text from 'shakespeare-hamlet.txt'. Total characters: {len(hamlet_text)}")
print("\nFirst 500 characters:")
print(hamlet_text[:500])

Loaded text from 'shakespeare-hamlet.txt'. Total characters: 162881

First 500 characters:
[The Tragedie of Hamlet by William Shakespeare 1599]


Actus Primus. Scoena Prima.

Enter Barnardo and Francisco two Centinels.

  Barnardo. Who's there?
  Fran. Nay answer me: Stand & vnfold
your selfe

   Bar. Long liue the King

   Fran. Barnardo?
  Bar. He

   Fran. You come most carefully vpon your houre

   Bar. 'Tis now strook twelue, get thee to bed Francisco

   Fran. For this releefe much thankes: 'Tis bitter cold,
And I am sicke at heart

   Barn. Haue you had quiet Guard?
  Fran. Not


## 2. Tokenization: Converting Text to Numerical IDs

Tokenization is the process of breaking down text into smaller units (tokens). For our LLM, we'll create a vocabulary of unique words and map them to integer IDs. This allows the model to process the text numerically.

In [13]:
# Import necessary NLTK module for tokenization
from nltk.tokenize import word_tokenize

# Download the 'punkt' tokenizer models
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True) # Added to download punkt_tab

# Convert text to lowercase and tokenize into words
words = word_tokenize(hamlet_text.lower())

# Create a vocabulary: map each unique word to an integer ID
# We'll reserve 0 for padding or unknown tokens if needed later
unique_words = sorted(list(set(words)))
word_to_int = {word: i + 1 for i, word in enumerate(unique_words)}
int_to_word = {i + 1: word for i, word in enumerate(unique_words)}

vocabulary_size = len(unique_words) + 1 # +1 for padding/unknown

# Convert the words in the text to their corresponding integer IDs
tokenized_text = [word_to_int[word] for word in words]

print(f"Total words in text: {len(words)}")
print(f"Unique words (vocabulary size): {vocabulary_size - 1}")
print(f"First 10 words: {words[:10]}")
print(f"First 10 token IDs: {tokenized_text[:10]}")
print("\nSample word-to-int mappings:")
# Print a few sample mappings to show how words map to integers
for i, (word, idx) in enumerate(word_to_int.items()):
    if i >= 10: # Limit to first 10 for display
        break
    print(f"  '{word}': {idx}")

Total words in text: 36409
Unique words (vocabulary size): 4790
First 10 words: ['[', 'the', 'tragedie', 'of', 'hamlet', 'by', 'william', 'shakespeare', '1599', ']']
First 10 token IDs: [27, 4158, 4272, 2824, 1847, 596, 4657, 3687, 23, 28]

Sample word-to-int mappings:
  '!': 1
  '&': 2
  ''': 3
  ''d': 4
  ''em': 5
  ''gainst': 6
  ''m': 7
  ''re': 8
  ''s': 9
  ''t': 10


## 3. Model Architecture: Defining a Simple RNN

For a simple LLM, we'll build a recurrent neural network (RNN) using Keras (part of TensorFlow). Our model will consist of:

1.  **Embedding Layer:** This layer takes the integer-encoded words and maps them to dense vectors of fixed size. This helps in capturing semantic relationships between words.
2.  **LSTM Layer (Long Short-Term Memory):** A type of RNN layer particularly good at learning long-term dependencies in sequences, which is crucial for language models. LSTM helps in understanding context.
3.  **Dense Output Layer:** This final layer will output a probability distribution over the entire vocabulary for the next word in the sequence. We'll use a `softmax` activation function for this.

In [15]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Define model parameters
embedding_dim = 128  # Dimension of the word embeddings
lstm_units = 256     # Number of LSTM units
sequence_length = 50 # Max number of words in an input sequence

# Build the sequential model
model = Sequential([
    Embedding(vocabulary_size, embedding_dim, input_shape=(sequence_length,)), # Use input_shape for fixed sequence size
    LSTM(lstm_units),
    Dense(vocabulary_size, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Display the model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)